Cell 1 — Settings and packages

In [1]:
DATE = None     # 指定美股交易日，例如 "2026-11-23"；None = 最近一個已收盤交易日
LIVE = None     # None = 自動判斷（美股交易中→盤中快照，收盤後→收盤價）；也可手動設 True／False

import io, logging, warnings, datetime as dt
from concurrent.futures import ThreadPoolExecutor
from zoneinfo import ZoneInfo
import pandas as pd
import requests
import yfinance as yf
from yfinance.data import YfData

# yfinance 在 pandas 3.x 下每抓一檔就噴一次 Pandas4Warning，抓不到的代碼還會印錯誤日誌；
# 全部靜音，缺資料改由 Cell 3 的「無資料」清單回報。
warnings.filterwarnings("ignore", module="yfinance")
logging.getLogger("yfinance").setLevel(logging.CRITICAL)

NY = ZoneInfo("America/New_York")
print("yfinance", yf.__version__, "｜pandas", pd.__version__)

yfinance 1.7.0 ｜pandas 3.1.1


Cell 2 — Tickers

In [2]:
# (顯示名稱, Yahoo 代碼, 收盤小數位, 漲跌點數小數位)
INDEX_ROWS = [
    ("道瓊", "^DJI", 0, 2), ("標普", "^GSPC", 0, 2), ("納指", "^IXIC", 0, 2),
    ("費半", "^SOX", 0, 2), ("VIX ", "^VIX", 2, 1),
    ("台積", "TSM", 2, 2), ("聯電", "UMC", 2, 2), ("日月光", "ASX", 2, 2),
]
# S&P 500 類股指數（抓不到時改用 SPDR ETF）
SECTORS = {
    "資訊科技": ("^SP500-45", "XLK"), "通訊服務": ("^SP500-50", "XLC"),
    "非必需消費": ("^SP500-25", "XLY"), "必需消費": ("^SP500-30", "XLP"),
    "能源": ("^SP500-10", "XLE"), "原材料": ("^SP500-15", "XLB"),
    "工業": ("^SP500-20", "XLI"), "醫療保健": ("^SP500-35", "XLV"),
    "金融": ("^SP500-40", "XLF"), "公用事業": ("^SP500-55", "XLU"),
    "不動產": ("^SP500-60", "XLRE"),
}
FOCUS = {
    "NVIDIA": "NVDA", "SuperMicro": "SMCI", "Tesla": "TSLA", "AMD": "AMD",
    "Meta": "META", "Google": "GOOGL", "Morgan Stanley": "MS", "Goldman Sachs": "GS",
    "Amazon": "AMZN", "JP Morgan": "JPM", "Apple": "AAPL", "Microsoft": "MSFT",
}
MACRO_ROWS = [("美元", "DX-Y.NYB", 2), ("黃金", "GC=F", 2), ("西德州", "CL=F", 2), ("布蘭特", "BZ=F", 2)]
# 外匯（寫美元段落用）：(名稱, 代碼, 是否為「美元兌該貨幣」報價)
FX_ROWS = [("歐元", "EURUSD=X", False), ("英鎊", "GBPUSD=X", False), ("日圓", "JPY=X", True),
           ("澳幣", "AUDUSD=X", False), ("人民幣", "CNY=X", True)]

TICKERS = ([t for _, t, _, _ in INDEX_ROWS] + [v[0] for v in SECTORS.values()]
           + [v[1] for v in SECTORS.values()] + list(FOCUS.values())
           + [t for _, t, _ in MACRO_ROWS] + [t for _, t, _ in FX_ROWS] + ["^TNX", "^IRX", "^TYX"])
print(len(TICKERS), "檔代碼")

54 檔代碼


Cell 3 — Prices

In [3]:
_YF = YfData()                                   # 共用 yfinance 的 session／cookie
_SPARK = "https://query1.finance.yahoo.com/v8/finance/spark"
_RANGES = [("1mo", 28), ("3mo", 89), ("6mo", 180), ("1y", 360), ("2y", 725),
           ("5y", 1820), ("10y", 3640), ("max", 10**9)]

def _spark(chunk, rng):
    """一次抓 ≤20 檔日收盤；Yahoo 沒有的代碼會直接略過。請求失敗回傳 None。"""
    try:
        j = _YF.get_raw_json(_SPARK, params={"symbols": ",".join(chunk), "range": rng, "interval": "1d"})
    except Exception:
        return None
    # 日 K 時間戳為該市場開盤時刻（美股 14:30、期貨 05:00、外匯 00:00 UTC），取 UTC 日期即為交易日
    return {s: pd.Series(v["close"], index=pd.to_datetime(v["timestamp"], unit="s").normalize(), dtype=float)
            for s, v in j.items() if v.get("timestamp") and v.get("close")}

def download_closes(tickers, since):
    """回傳 since 以後的日收盤價（欄＝代碼）。
    先用 Yahoo spark 批次端點（每批 20 檔、並行，500 檔約 1 秒）；整批請求失敗的才退回 yf.download（每檔一個請求）。"""
    tickers = list(dict.fromkeys(tickers))
    since = pd.Timestamp(since).normalize()
    rng = next(r for r, d in _RANGES if d >= (pd.Timestamp.now().normalize() - since).days)
    chunks = [tickers[i:i + 20] for i in range(0, len(tickers), 20)]
    out, retry = {}, []
    with ThreadPoolExecutor(8) as ex:
        for chunk, res in zip(chunks, ex.map(lambda c: _spark(c, rng), chunks)):
            if res is None:
                retry += chunk
            else:
                out.update(res)
    if retry:
        df = yf.download(retry, start=since, interval="1d", auto_adjust=False,
                         progress=False, threads=True)["Close"]
        if isinstance(df, pd.Series):
            df = df.to_frame(retry[0])
        df.index = pd.to_datetime([d.date() for d in df.index])
        out.update({t: df[t] for t in df.columns})
    close = pd.DataFrame(out).sort_index()
    return close[close.index >= since]

d0 = pd.Timestamp(DATE).normalize() if DATE else pd.Timestamp.now(NY).tz_localize(None).normalize()
closes = download_closes(TICKERS, since=d0 - pd.Timedelta(days=20))

spx = closes["^GSPC"].dropna()
td = d0 if DATE else spx.index[-1]
now_ny = dt.datetime.now(NY)
# 美股正在交易：平日 09:30–16:00（美東），且今天已有價格
market_open = (now_ny.weekday() < 5 and dt.time(9, 30) <= now_ny.time() < dt.time(16, 0)
               and spx.index[-1].date() == now_ny.date())
if LIVE is None:
    LIVE = (not DATE) and market_open       # 04:25 台北 → 盤中快照；05:08 台北 → 收盤價
if not DATE and market_open and not LIVE:
    td = spx.index[-2]                      # 指定不要盤中價 → 用前一個交易日收盤
print(f"現在美東時間 {now_ny:%m/%d %H:%M}｜交易日：{td.date()}",
      "｜⚠ 盤中快照，非收盤價" if LIVE else "｜收盤價")

def chg(t, strict=True):
    """回傳 (收盤, 漲跌點數, 漲跌幅%)；strict=False 允許用交易日前最後一筆（期貨、外匯）。"""
    if t not in closes.columns:
        return None
    s = closes[t].dropna()
    s = s[s.index <= td]
    if len(s) < 2 or (strict and s.index[-1] != td):
        return None
    last, prev = float(s.iloc[-1]), float(s.iloc[-2])
    return last, last - prev, (last / prev - 1) * 100

# 類股指數抓不到時 Cell 5 會自動改用 ETF，不列入缺資料
_has_fallback = {idx for idx, etf in SECTORS.values() if chg(etf) is not None}
missing = [t for t in TICKERS if t not in _has_fallback and chg(t, strict=False) is None]
print("無資料：", missing if missing else "無")

現在美東時間 11/24 03:34｜交易日：2026-11-23 ｜收盤價
無資料： 無


Cell 4 — 《股市》 indices and ADRs

In [4]:
index_lines = []
for label, t, cdec, chdec in INDEX_ROWS:
    r = chg(t)
    if r:
        c, d, p = r
        index_lines.append(f"🔘{label} {'漲' if d >= 0 else '跌'}{abs(d):.{chdec}f} ({p:+.2f}%) 收 {c:.{cdec}f}")
    else:
        index_lines.append(f"🔘{label} （{t} 無資料，請手動補）")
print("\n\n".join(index_lines))

🔘道瓊 漲1447.98 (+3.01%) 收 49595

🔘標普 漲326.78 (+4.73%) 收 7237

🔘納指 漲1471.83 (+6.51%) 收 24090

🔘費半 漲1179.46 (+14.03%) 收 9588

🔘VIX  跌9.3 (-25.83%) 收 26.62

🔘台積 漲44.89 (+14.08%) 收 363.60

🔘聯電 漲2.35 (+19.52%) 收 14.39

🔘日月光 漲5.61 (+21.36%) 收 31.88


Cell 5 — Strongest and weakest sectors

In [5]:
rows = []
for zh, (idx, etf) in SECTORS.items():
    r, src = chg(idx), idx
    if r is None:
        r, src = chg(etf), etf
    if r:
        rows.append((zh, round(r[2], 2), src))
sector_df = pd.DataFrame(rows, columns=["sector", "pct", "source"]).sort_values("pct", ascending=False)
strong, weak = sector_df.head(3), sector_df.sort_values("pct").head(3)
sector_text = ("🔹強勢類股：\n\n" + "\n\n".join(f"{r.sector}  {r.pct:+.2f}%" for r in strong.itertuples())
               + "\n\n🔹弱勢類股：\n\n" + "\n\n".join(f"{r.sector}  {r.pct:+.2f}%" for r in weak.itertuples()))
display(sector_df.reset_index(drop=True))
print(sector_text)

,sector,pct,source
0,資訊科技,8.97,XLK
1,非必需消費,3.97,XLY
2,工業,3.90,XLI
3,通訊服務,3.57,XLC
4,原材料,3.23,XLB
5,金融,1.98,XLF
6,公用事業,1.48,XLU
7,醫療保健,1.23,XLV
8,不動產,1.08,^SP500-60
9,必需消費,-0.09,XLP


🔹強勢類股：

資訊科技  +8.97%

非必需消費  +3.97%

工業  +3.90%

🔹弱勢類股：

能源  -1.06%

必需消費  -0.09%

不動產  +1.08%


Cell 6 — 焦點股 (sorted by 漲幅, highest first)

In [6]:
focus_df = (pd.DataFrame([(n, t, r[0], r[2]) for n, t in FOCUS.items() if (r := chg(t))],
                         columns=["name", "ticker", "close", "pct"])
            .sort_values("pct", ascending=False).reset_index(drop=True))
display(focus_df.round(2))
focus_line = "🔹焦點股： " + "、".join(f"{r.name} {r.pct:+.2f}%" for r in focus_df.itertuples())
print(focus_line)

,name,ticker,close,pct
0,NVIDIA,NVDA,165.05,31.48
1,SuperMicro,SMCI,24.53,13.30
2,Tesla,TSLA,385.46,11.46
3,AMD,AMD,453.46,6.85
4,Meta,META,597.60,6.55
5,Google,GOOGL,282.10,6.34
6,Morgan Stanley,MS,172.60,4.18
7,Goldman Sachs,GS,823.62,4.16
8,Amazon,AMZN,255.39,3.47
9,JP Morgan,JPM,316.66,1.97


🔹焦點股： NVIDIA +31.48%、SuperMicro +13.30%、Tesla +11.46%、AMD +6.85%、Meta +6.55%、Google +6.34%、Morgan Stanley +4.18%、Goldman Sachs +4.16%、Amazon +3.47%、JP Morgan +1.97%、Apple +1.90%、Microsoft +1.65%


Cell 7 — 《美債/美元/黃金/原油》, plus yield and FX notes for writing

In [7]:
r = chg("^TNX", strict=False)
if r:
    scale = 10 if r[0] > 20 else 1          # 舊式報價為殖利率×10
    bond_line = f"🔘美債 {round(r[1] / scale * 100):+d}bps 收{r[0] / scale:.2f}%"
else:
    bond_line = "🔘美債 （無資料，請手動補）"
macro_lines = [bond_line]
for label, t, dec in MACRO_ROWS:
    r = chg(t, strict=False)
    macro_lines.append(f"🔘{label} {r[2]:+.2f}% 收 {r[0]:.{dec}f}" if r else f"🔘{label} （{t} 無資料，請手動補）")
print("\n\n".join(macro_lines))

# 寫作參考：殖利率曲線（Yahoo：^IRX 13週、^TNX 10年、^TYX 30年）與外匯
print("\n【寫作參考】")
for name, t in [("13週", "^IRX"), ("10年", "^TNX"), ("30年", "^TYX")]:
    r = chg(t, strict=False)
    if r:
        s = 10 if r[0] > 20 else 1
        print(f"  {name}殖利率 {r[0] / s:.2f}% ({round(r[1] / s * 100):+d}bp)")
fx = []
for name, t, usd_base in FX_ROWS:
    r = chg(t, strict=False)
    if r:
        p = -r[2] if usd_base else r[2]      # 換算成「該貨幣對美元」的升貶
        fx.append(f"{name}{'升值' if p >= 0 else '貶值'}{abs(p):.2f}%")
print("  G10/亞幣對美元：" + "、".join(fx))

🔘美債 +1bps 收4.92%

🔘美元 +0.49% 收 104.32

🔘黃金 +3.87% 收 3651.40

🔘西德州 -4.54% 收 80.17

🔘布蘭特 -3.46% 收 89.37

【寫作參考】
  13週殖利率 4.15% (+3bp)
  10年殖利率 4.92% (+1bp)
  30年殖利率 5.29% (-1bp)
  G10/亞幣對美元：歐元升值0.02%、英鎊升值0.07%、日圓升值0.49%、澳幣貶值0.23%、人民幣貶值0.16%


Cell 8 — (Optional) S&P 500 breadth and each sector's leaders and laggards

In [8]:
GICS_ZH = {"Information Technology": "資訊科技", "Communication Services": "通訊服務",
           "Consumer Discretionary": "非必需消費", "Consumer Staples": "必需消費", "Energy": "能源",
           "Materials": "原材料", "Industrials": "工業", "Health Care": "醫療保健", "Financials": "金融",
           "Utilities": "公用事業", "Real Estate": "不動產"}

wiki = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies",
                    headers={"User-Agent": "Mozilla/5.0"}, timeout=30).text
cons = pd.read_html(io.StringIO(wiki), attrs={"id": "constituents"})[0]
cons = pd.DataFrame({"ticker": cons["Symbol"].str.replace(".", "-", regex=False),
                     "name": cons["Security"], "sector": cons["GICS Sector"].map(GICS_ZH)})

sp = download_closes(cons["ticker"], since=td - pd.Timedelta(days=15))
sp = sp[sp.index <= td].dropna(axis=1, how="all")
ret = (sp.iloc[-1] / sp.iloc[-2] - 1) * 100
sp_df = (cons.assign(pct=cons["ticker"].map(ret)).dropna(subset=["pct"])
             .sort_values("pct", ascending=False, ignore_index=True))
sp_df["label"] = sp_df["name"] + " " + sp_df["pct"].map("{:+.1f}%".format)

def _summary(g):
    up, down = int((g["pct"] > 0).sum()), int((g["pct"] < 0).sum())
    return {"上漲": up, "下跌": down, "上漲占比": up / len(g) * 100,
            "領漲": "、".join(g["label"].head(3)), "領跌": "、".join(g["label"].tail(3)[::-1])}

sector_pct = sector_df.set_index("sector")["pct"]
spx_r = chg("^GSPC")
breadth_table = pd.DataFrame(
    {"S&P 500": {"漲跌%": spx_r[2] if spx_r else float("nan"), **_summary(sp_df)},
     **{sec: {"漲跌%": sector_pct.get(sec, float("nan")), **_summary(g)}
        for sec, g in sp_df.groupby("sector")}}).T.infer_objects()
breadth_table = pd.concat([breadth_table.iloc[:1],
                           breadth_table.iloc[1:].sort_values("漲跌%", ascending=False)])
breadth_table.index.name = "類股"

display(breadth_table.style
        .format({"漲跌%": "{:+.2f}%", "上漲占比": "{:.1f}%", "上漲": "{:d}", "下跌": "{:d}"})
        .set_properties(subset=["領漲", "領跌"], **{"text-align": "left", "white-space": "normal"})
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
        .set_caption(f"S&P 500 市場廣度與各類股領漲／領跌（{td:%Y/%m/%d}；"
                     f"漲跌家數比 {(sp_df['pct'] > 0).sum() / max((sp_df['pct'] < 0).sum(), 1):.2f}）"))

,漲跌%,上漲,下跌,上漲占比,領漲,領跌
類股,,,,,,
S&P 500,+4.73%,419,83,83.3%,Nvidia +31.5%、Dell Technologies +20.5%、Micron Technology +19.0%,CrowdStrike -8.9%、International Paper -4.5%、Elevance Health -3.8%
資訊科技,+8.97%,74,2,97.4%,Nvidia +31.5%、Dell Technologies +20.5%、Micron Technology +19.0%,CrowdStrike -8.9%、Roper Technologies -0.4%、PTC Inc. +0.5%
非必需消費,+3.97%,44,3,93.6%,"Tesla, Inc. +11.5%、Aptiv +7.3%、Norwegian Cruise Line Holdings +6.3%",Tractor Supply -2.3%、Best Buy -1.0%、TJX Companies -0.3%
工業,+3.90%,80,3,96.4%,Bloom Energy +18.1%、Vertiv +12.1%、Comfort Systems USA +11.1%,FedEx Freight -1.5%、Verisk Analytics -1.1%、Paychex -0.1%
通訊服務,+3.57%,17,5,77.3%,AppLovin +7.4%、Meta Platforms +6.6%、Alphabet Inc. (Class A) +6.3%,Comcast -0.7%、Charter Communications -0.6%、Take-Two Interactive -0.3%
原材料,+3.23%,21,3,87.5%,Freeport-McMoRan +8.1%、Mosaic Company (The) +7.7%、Newmont +6.4%,International Paper -4.5%、LyondellBasell -2.0%、Dow Inc. -1.5%
金融,+1.98%,64,12,84.2%,Robinhood Markets +8.4%、Interactive Brokers +7.4%、Invesco +6.9%,Arthur J. Gallagher & Co. -1.5%、Erie Indemnity -1.4%、CME Group -1.0%
公用事業,+1.48%,27,4,87.1%,NRG Energy +11.3%、Vistra Corp. +7.5%、Constellation Energy +6.0%,American Water Works -1.5%、Consolidated Edison -0.7%、Duke Energy -0.2%
醫療保健,+1.23%,48,11,80.0%,"Moderna +8.8%、Charles River Laboratories +7.6%、Illumina, Inc. +6.5%",Elevance Health -3.8%、McKesson -2.0%、Humana -1.6%


Cell 9 — Full block to paste into the report

In [9]:
text = ("⚠ 盤中快照（非收盤價）\n\n" if LIVE else "") + (
    f"🇺🇸美股盤後(完整版)\n\n{td:%Y/%m/%d}\n\n《股市》\n\n" + "\n\n".join(index_lines)
    + "\n\n" + sector_text + "\n\n" + focus_line
    + "\n\n《美債/美元/黃金/原油》\n\n" + "\n\n".join(macro_lines) + "\n")
print(text)

🇺🇸美股盤後(完整版)

2026/11/23

《股市》

🔘道瓊 漲1447.98 (+3.01%) 收 49595

🔘標普 漲326.78 (+4.73%) 收 7237

🔘納指 漲1471.83 (+6.51%) 收 24090

🔘費半 漲1179.46 (+14.03%) 收 9588

🔘VIX  跌9.3 (-25.83%) 收 26.62

🔘台積 漲44.89 (+14.08%) 收 363.60

🔘聯電 漲2.35 (+19.52%) 收 14.39

🔘日月光 漲5.61 (+21.36%) 收 31.88

🔹強勢類股：

資訊科技  +8.97%

非必需消費  +3.97%

工業  +3.90%

🔹弱勢類股：

能源  -1.06%

必需消費  -0.09%

不動產  +1.08%

🔹焦點股： NVIDIA +31.48%、SuperMicro +13.30%、Tesla +11.46%、AMD +6.85%、Meta +6.55%、Google +6.34%、Morgan Stanley +4.18%、Goldman Sachs +4.16%、Amazon +3.47%、JP Morgan +1.97%、Apple +1.90%、Microsoft +1.65%

《美債/美元/黃金/原油》

🔘美債 +1bps 收4.92%

🔘美元 +0.49% 收 104.32

🔘黃金 +3.87% 收 3651.40

🔘西德州 -4.54% 收 80.17

🔘布蘭特 -3.46% 收 89.37

